# 02 — Image Processing

Preprocessing pipeline:

```
CDL (USA-wide, EPSG:5070, 30 m)
  └─ Step 1: Reproject EPSG:5070 → EPSG:4326  +  clip to study area  +  resample to S2 res
  └─ Step 2: Filter CDL classes (keep target crops only)

S2 images (EPSG:4326, ~10 m, study area)
  └─ Step 3: Assign NoData  (invalid / negative / NaN → -9999, cast to float32)

Verify: CDL and S2 share the same CRS, bounds, and pixel grid
```

> **Why CDL first?**  The CDL raster covers the entire USA.  
> Reprojecting + clipping it to the study area *before* any S2 work keeps all subsequent operations cheap.

## Configuration

In [1]:
import sys, os
from glob import glob
sys.path.append("../")

# ── S2 source directory (Google Drive mount) ──────────────────────────────────
S2_DIR = "/Users/dikaizm/Library/CloudStorage/GoogleDrive-dikamaah@gmail.com/My Drive/S2_Annual_15d_sacramento_3"

# All S2 files for all three years (training: 2022+2023, test: 2024)
S2_ALL = sorted(glob(os.path.join(S2_DIR, "S2H_20*.tif")))
# Split by year for CDL alignment
S2_BY_YEAR = {}
for p in S2_ALL:
    fname = os.path.basename(p)
    year = fname.split("_")[1]   # "S2H_2022_..." → "2022"
    S2_BY_YEAR.setdefault(year, []).append(p)

print(f"S2 files found:")
for yr, files in sorted(S2_BY_YEAR.items()):
    print(f"  {yr}: {len(files)} files")

# ── CDL labels — one per year (external drive) ────────────────────────────────
CDL_RAW = {
    "2022": "/Volumes/T7/research-crop-mapping-geoai/data/cdl/2022_30m_cdls/2022_30m_cdls.tif",
    "2023": "/Volumes/T7/research-crop-mapping-geoai/data/cdl/2023_30m_cdls/2023_30m_cdls.tif",
    "2024": "/Volumes/T7/research-crop-mapping-geoai/data/cdl/2024_30m_cdls/2024_30m_cdls.tif",
}

# ── Output directories (external drive) ───────────────────────────────────────
OUT_DIR     = "/Volumes/T7/research-crop-mapping-geoai/data/processed"
OUT_CDL_DIR = os.path.join(OUT_DIR, "cdl")
OUT_S2_DIR  = os.path.join(OUT_DIR, "s2")
os.makedirs(OUT_CDL_DIR, exist_ok=True)
os.makedirs(OUT_S2_DIR,  exist_ok=True)

# Per-year CDL output paths
CDL_REPROJECTED = {yr: os.path.join(OUT_CDL_DIR, f"cdl_{yr}_study_area.tif")         for yr in CDL_RAW}
CDL_FILTERED    = {yr: os.path.join(OUT_CDL_DIR, f"cdl_{yr}_study_area_filtered.tif") for yr in CDL_RAW}

# ── Parameters ────────────────────────────────────────────────────────────────
S2_NODATA_VALUE  = -9999.0
CDL_NODATA_VALUE = 0

# 10 active crop classes (≥1% area). Fallow/Idle (61) → background.
KEEP_CLASSES = [3, 6, 24, 36, 37, 54, 69, 75, 76, 220]
#   3=Rice, 6=Sunflower, 24=Winter Wheat, 36=Alfalfa, 37=Other Hay,
#   54=Tomatoes, 69=Grapes, 75=Almonds, 76=Walnuts, 220=Plums

print(f"\nKEEP_CLASSES ({len(KEEP_CLASSES)}): {KEEP_CLASSES}")
print(f"OUT_DIR: {OUT_DIR}")
for yr, raw in CDL_RAW.items():
    print(f"CDL {yr}: {'✅' if os.path.exists(raw) else '❌'} {raw}")

S2 files found:
  2022: 25 files
  2023: 25 files
  2024: 25 files

KEEP_CLASSES (10): [3, 6, 24, 36, 37, 54, 69, 75, 76, 220]
OUT_DIR: /Volumes/T7/research-crop-mapping-geoai/data/processed
CDL 2022: ✅ /Volumes/T7/research-crop-mapping-geoai/data/cdl/2022_30m_cdls/2022_30m_cdls.tif
CDL 2023: ✅ /Volumes/T7/research-crop-mapping-geoai/data/cdl/2023_30m_cdls/2023_30m_cdls.tif
CDL 2024: ✅ /Volumes/T7/research-crop-mapping-geoai/data/cdl/2024_30m_cdls/2024_30m_cdls.tif


### (Optional) Download S2 samples via gdown
Run this cell only if you need to download the files instead of reading from Google Drive mount.

In [2]:
# Uncomment and run to download via gdown
# import gdown, re
#
# GDRIVE_LINKS = [
#     "https://drive.google.com/open?id=1iyWjuXqUWc6oK_jAh99_JkIc8mzwCpjf&usp=drive_fs",
#     "https://drive.google.com/open?id=1u7iXqd2cNWHkdJIRS2U52J3SC10FfZWy&usp=drive_fs",
#     "https://drive.google.com/open?id=1mFGEMKdzTvMB-AKPz8MQceWpWCz3E6fa&usp=drive_fs",
# ]
#
# def _extract_id(url):
#     m = re.search(r'[?&]id=([a-zA-Z0-9_-]+)', url)
#     return m.group(1) if m else None
#
# os.makedirs(OUT_S2_DIR, exist_ok=True)
# for url in GDRIVE_LINKS:
#     fid = _extract_id(url)
#     out = gdown.download(id=fid, output=OUT_S2_DIR + "/", quiet=False, resume=True, fuzzy=True)
#     print(f"Downloaded: {out}")

---
## Inspect: CRS Mismatch
Before processing, confirm the coordinate system difference between CDL and S2.

In [3]:
import rasterio

print("=" * 55)
with rasterio.open(S2_ALL[0]) as src:
    s2_crs    = src.crs
    s2_bounds = src.bounds
    s2_res    = src.res
    s2_shape  = (src.width, src.height)
    s2_bands  = src.count
    print(f"S2  ({os.path.basename(S2_ALL[0])})")
    print(f"  CRS       : {s2_crs}")
    print(f"  Bounds    : {s2_bounds}")
    print(f"  Resolution: {s2_res}")
    print(f"  Size      : {s2_shape[0]} x {s2_shape[1]}  |  {s2_bands} bands")

print()
with rasterio.open(CDL_RAW["2022"]) as src:
    cdl_crs    = src.crs
    cdl_bounds = src.bounds
    cdl_res    = src.res
    cdl_shape  = (src.width, src.height)
    print(f"CDL ({os.path.basename(CDL_RAW['2022'])})")
    print(f"  CRS       : {cdl_crs}")
    print(f"  Bounds    : {cdl_bounds}")
    print(f"  Resolution: {cdl_res} m")
    print(f"  Size      : {cdl_shape[0]} x {cdl_shape[1]}")

print()
print(f"CRS match?  {'✅ Yes' if s2_crs == cdl_crs else '❌ No — reprojection required'}")
print("=" * 55)

S2  (S2H_2022_2022_01_01.tif)
  CRS       : EPSG:4326
  Bounds    : BoundingBox(left=-122.07799283987697, bottom=38.753860346086626, right=-121.57529560688369, top=39.17463122516821)
  Resolution: (8.983152841195215e-05, 8.983152841195215e-05)
  Size      : 5596 x 4684  |  11 bands

CDL (2022_30m_cdls.tif)
  CRS       : EPSG:5070
  Bounds    : BoundingBox(left=-2356095.0, bottom=276915.0, right=2258235.0, top=3172605.0)
  Resolution: (30.0, 30.0) m
  Size      : 153811 x 96523

CRS match?  ❌ No — reprojection required


---
## Step 1 — Reproject, Clip & Resample CDL

Three operations in **one pass** using the S2 image as the reference grid:

| | CDL before | CDL after |
|---|---|---|
| CRS | EPSG:5070 (Conus Albers) | EPSG:4326 (WGS84) |
| Coverage | Entire USA | Study area only |
| Resolution | 30 m | ~10 m (matches S2) |

Using `Resampling.nearest` to preserve integer class labels.

In [4]:
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

# Use any S2 file as the reference grid (all S2 share the same grid/CRS/bounds)
S2_REF = S2_ALL[0]
with rasterio.open(S2_REF) as s2_ref:
    target_crs       = s2_ref.crs
    target_transform = s2_ref.transform
    target_width     = s2_ref.width
    target_height    = s2_ref.height

print(f"Reference grid: {target_width}×{target_height}  CRS={target_crs}\n")

for yr, cdl_raw_path in CDL_RAW.items():
    out_path = CDL_REPROJECTED[yr]
    if os.path.exists(out_path):
        print(f"[{yr}] ✅ Already exists: {os.path.basename(out_path)}")
        continue
    if not os.path.exists(cdl_raw_path):
        print(f"[{yr}] ❌ CDL not found: {cdl_raw_path}  — SKIPPING")
        continue

    with rasterio.open(cdl_raw_path) as cdl_src:
        dst_data = np.zeros((1, target_height, target_width), dtype=np.uint8)
        reproject(
            source=rasterio.band(cdl_src, 1),
            destination=dst_data,
            src_transform=cdl_src.transform,
            src_crs=cdl_src.crs,
            dst_transform=target_transform,
            dst_crs=target_crs,
            resampling=Resampling.nearest,
        )

    profile = {
        "driver": "GTiff", "dtype": "uint8", "nodata": 0,
        "width": target_width, "height": target_height, "count": 1,
        "crs": target_crs, "transform": target_transform, "compress": "lzw",
    }
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(dst_data)
    print(f"[{yr}] ✅ Reprojected CDL → {os.path.basename(out_path)}")


Reference grid: 5596×4684  CRS=EPSG:4326

[2022] ✅ Already exists: cdl_2022_study_area.tif
[2023] ✅ Already exists: cdl_2023_study_area.tif
[2024] ✅ Already exists: cdl_2024_study_area.tif


---
## Step 2 — Filter CDL Classes
Keep only the target crop classes; all other pixels are set to 0 (background).

In [5]:
from utils import label as label_utils
from utils.constants import USDA_CDL_NAMES

for yr in CDL_RAW:
    in_path  = CDL_REPROJECTED[yr]
    out_path = CDL_FILTERED[yr]

    if not os.path.exists(in_path):
        print(f"[{yr}] ❌ Reprojected CDL not found — run Step 1 first")
        continue
    if os.path.exists(out_path):
        print(f"[{yr}] ✅ Already filtered: {os.path.basename(out_path)}")
        continue

    label_utils.label_filtering(in_path=in_path, out_path=out_path, keep_classes=KEEP_CLASSES)

    with rasterio.open(out_path) as src:
        data = src.read(1)
        unique, counts = np.unique(data, return_counts=True)
        total = data.size
        print(f"\n[{yr}] Class distribution in filtered CDL:")
        for cls, cnt in zip(unique, counts):
            if cls == 0: continue
            name = USDA_CDL_NAMES.get(int(cls), "unknown")
            print(f"  {cls:>4}  {name:<30}  {cnt:>12,}  {cnt/total*100:>5.2f}%")


[2022] ✅ Already filtered: cdl_2022_study_area_filtered.tif
[2023] ✅ Already filtered: cdl_2023_study_area_filtered.tif
[2024] ✅ Already filtered: cdl_2024_study_area_filtered.tif


---
## Step 3 — Assign NoData: S2 Images
Loop over all 3 sample images.  Replace negative, NaN, and Inf values with the nodata sentinel and cast to float32.

In [6]:
S2_PROCESSED_PATHS = []

for src_path in S2_ALL:
    fname    = os.path.basename(src_path)
    out_path = os.path.join(OUT_S2_DIR, fname.replace(".tif", "_processed.tif"))

    if os.path.exists(out_path):
        S2_PROCESSED_PATHS.append(out_path)
        continue

    with rasterio.open(src_path) as src:
        profile = src.profile.copy()
        profile.update(dtype="float32", nodata=S2_NODATA_VALUE, compress="lzw")
        data = src.read().astype(np.float32)

    invalid = (data < 0) | np.isnan(data) | np.isinf(data)
    data[invalid] = S2_NODATA_VALUE

    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(data)

    S2_PROCESSED_PATHS.append(out_path)
    print(f"✅ {fname}  invalid_px={invalid.sum():,}")

print(f"\nTotal processed S2 files: {len(S2_PROCESSED_PATHS)}")
by_year = {}
for p in S2_PROCESSED_PATHS:
    yr = os.path.basename(p).split("_")[1]
    by_year.setdefault(yr, []).append(p)
for yr, files in sorted(by_year.items()):
    print(f"  {yr}: {len(files)} files")


✅ S2H_2024_2024_03_31.tif  invalid_px=361,079
✅ S2H_2024_2024_04_15.tif  invalid_px=655,897
✅ S2H_2024_2024_04_30.tif  invalid_px=951,940
✅ S2H_2024_2024_05_15.tif  invalid_px=890,572
✅ S2H_2024_2024_05_30.tif  invalid_px=1,341,164
✅ S2H_2024_2024_06_14.tif  invalid_px=1,436,732
✅ S2H_2024_2024_06_29.tif  invalid_px=1,660,318
✅ S2H_2024_2024_07_14.tif  invalid_px=2,061,587
✅ S2H_2024_2024_07_29.tif  invalid_px=1,324,117
✅ S2H_2024_2024_08_13.tif  invalid_px=815,947
✅ S2H_2024_2024_08_28.tif  invalid_px=784,058
✅ S2H_2024_2024_09_12.tif  invalid_px=1,119,921
✅ S2H_2024_2024_09_27.tif  invalid_px=831,622
✅ S2H_2024_2024_10_12.tif  invalid_px=1,549,460
✅ S2H_2024_2024_10_27.tif  invalid_px=2,658,768
✅ S2H_2024_2024_11_11.tif  invalid_px=83,618,304
✅ S2H_2024_2024_11_26.tif  invalid_px=1,178,443
✅ S2H_2024_2024_12_11.tif  invalid_px=283,408,763
✅ S2H_2024_2024_12_26.tif  invalid_px=6,627,352

Total processed S2 files: 75
  2022: 25 files
  2023: 25 files
  2024: 25 files


---
## Verify Alignment
CDL and each S2 image must share the same CRS, bounds, and pixel dimensions.

In [7]:
# Verify a sample from each year aligns with its CDL
print(f"{'File':<50}  {'Shape':<14}  {'CDL match'}")
print("-" * 80)

for yr in sorted(S2_BY_YEAR):
    yr_files = [p for p in S2_PROCESSED_PATHS if os.path.basename(p).split('_')[1] == yr]
    cdl_path = CDL_FILTERED.get(yr)
    if not yr_files or not cdl_path or not os.path.exists(cdl_path):
        print(f"[{yr}] ⚠️  missing files")
        continue

    with rasterio.open(cdl_path) as cdl:
        cdl_shape = (cdl.width, cdl.height)
        cdl_crs   = cdl.crs

    with rasterio.open(yr_files[0]) as src:
        ok = (src.width, src.height) == cdl_shape and src.crs == cdl_crs
        print(f"[{yr}] {os.path.basename(yr_files[0]):<48}  {src.width}×{src.height:<6}  {'✅' if ok else '❌'}")

print("\nAlignment check complete.")


File                                                Shape           CDL match
--------------------------------------------------------------------------------
[2022] S2H_2022_2022_01_01_processed.tif                 5596×4684    ✅
[2023] S2H_2023_2023_01_01_processed.tif                 5596×4684    ✅
[2024] S2H_2024_2024_01_01_processed.tif                 5596×4684    ✅

Alignment check complete.
